In [10]:
import torch

In [11]:
torch.cuda.is_available()

True

In [12]:
class NpuStress(torch.nn.Module):
    LAYER_COUNT = 100

    def __init__(self):
        super(NpuStress, self).__init__()

        for i in range(self.LAYER_COUNT):
            self.__setattr__(f"linear{i}", torch.nn.Linear(1024, 1024))
            self.__setattr__(f"relu{i}", torch.nn.ReLU())

        self.final_linear = torch.nn.Linear(1024, 1024)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for _ in range(self.LAYER_COUNT):
            x = self.__getattr__(f"linear{_}")(x)
            x = self.__getattr__(f"relu{_}")(x)

        x = self.final_linear(x)
        return x

In [13]:
model = NpuStress()
model.eval()

NpuStress(
  (linear0): Linear(in_features=1024, out_features=1024, bias=True)
  (relu0): ReLU()
  (linear1): Linear(in_features=1024, out_features=1024, bias=True)
  (relu1): ReLU()
  (linear2): Linear(in_features=1024, out_features=1024, bias=True)
  (relu2): ReLU()
  (linear3): Linear(in_features=1024, out_features=1024, bias=True)
  (relu3): ReLU()
  (linear4): Linear(in_features=1024, out_features=1024, bias=True)
  (relu4): ReLU()
  (linear5): Linear(in_features=1024, out_features=1024, bias=True)
  (relu5): ReLU()
  (linear6): Linear(in_features=1024, out_features=1024, bias=True)
  (relu6): ReLU()
  (linear7): Linear(in_features=1024, out_features=1024, bias=True)
  (relu7): ReLU()
  (linear8): Linear(in_features=1024, out_features=1024, bias=True)
  (relu8): ReLU()
  (linear9): Linear(in_features=1024, out_features=1024, bias=True)
  (relu9): ReLU()
  (linear10): Linear(in_features=1024, out_features=1024, bias=True)
  (relu10): ReLU()
  (linear11): Linear(in_features=1024, ou

In [14]:
input_tensor = torch.randn(1000, 1024)

output = model(input_tensor)

output

tensor([[-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057],
        [-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057],
        [-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057],
        ...,
        [-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057],
        [-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057],
        [-0.0413,  0.0201,  0.0073,  ..., -0.0261,  0.0217, -0.0057]],
       grad_fn=<AddmmBackward0>)

In [15]:
traced_model = torch.jit.trace(model, input_tensor)

In [17]:
import qai_hub as hub
device = hub.Device("Dragonwing RB3 Gen 2 Vision Kit")

# https://workbench.aihub.qualcomm.com/docs/hub/api.html
calibration_data = dict(image=[torch.randn(1000, 1024).numpy() for _ in range(5)])

unquantized_compile_job = hub.submit_compile_job(
        model=traced_model,
        device=device,
        input_specs=dict(image=((1000, 1024), "float32")),
        options="--target_runtime onnx",
)

quantize_job = hub.submit_quantize_job(
        model=unquantized_compile_job.get_target_model(),
        calibration_data=calibration_data,
        weights_dtype=hub.QuantizeDtype.INT8,
        activations_dtype=hub.QuantizeDtype.INT8,
)

compile_job = hub.submit_compile_job(
        model=quantize_job.get_target_model(),
        device=device,
        options="--target_runtime qnn_context_binary --quantize_io",
)
target_model = compile_job.get_target_model()

Uploading tmpnb4i241g.pt


100%|██████████| 405M/405M [00:21<00:00, 19.8MB/s] 


Scheduled compile job (jgj07341p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgj07341p/

Waiting for compile job (jgj07341p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


Uploading dataset: 18.2MB [00:00, 20.7MB/s]                            27.0MB/s]


Scheduled quantize job (jp1dj14np) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp1dj14np/

Waiting for quantize job (jp1dj14np) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Scheduled compile job (jgdr34nkp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgdr34nkp/

Waiting for compile job (jgdr34nkp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


In [18]:
target_model.download("stress.bin")

stress.bin: 100%|██████████| 113M/113M [00:08<00:00, 13.5MB/s] 

Downloaded model to stress.bin


'stress.bin'